*Load relevant packages:*

In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run packages.ipynb
from gmsPython import nestingTree
# Local packages:
os.chdir(d['py'])
import mCGE
from valueShares import InitTree

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.


# CGE

The notebook starts from a database with conventional input data merged with material flows data and model parameters in the CGE model (specifying nesting trees, elasticities etc..). We set up a model instance and add all relevant modules one by one. For relevant modules, we also run small methods that help provide initial values for endogenous variables, and normalization parameters used in numerically tricky parts of the model. We then calibrate the model by first solving it without equilibrium constraints and then apply a relaxation algorithm to ensure equilibrium in a second step. Finally, the model and initial solution is packaged and stored in the data repository.

*Settings:*

In [2]:
t0 = 2019 # baseline year
v = 'vMain' # global name
name = f'{v}{t0}CGE' # name used when storing calibrated model instance

*Load:*

In [3]:
db = GpyDB(os.path.join(d['data'], f'{v}{t0}_db'), name = f'{name}_db', ws = d['work']) # load database
ws = db.ws # shorthand for the GamsWorkspace (we want to run everything from the same workspace)
dbIO = db.copy() # copy of database with input targets from the IO data

Initialize model:

In [4]:
M = mCGE.WasteManagementCGE(name, database = db) # initialize model instance

## 1. Modules

The class has prespecified methods for adding modules. The modules for this specific CGE are: Production, waste management, investment, consumer, government, trade, and emissions.

### A. Production

Production covers domestic production firms of non-waste outputs.

#### Nesting structure

*Nesting structure consists of two nesting "trees":*

In [5]:
nestInp = nestingTree.Tree('PInp', tree = db('nestProdInp').to_list(), f = 'CES_scaled')
nestOut = nestingTree.Tree('POut', tree = db('nestProdOut').to_list(), f= 'CET_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nestInp, nestOut]}, ws = ws)
nestAgg(namespace = {str(n)+'_input': n for n in db('n').union(db('nm_D'))})

*Add static user cost as initial value for prices on durables:*

In [6]:
db.aom(db('pD_dur'), name = 'pD', priority = 'first')

*Get initial values of $\mu$-parameters and intermediate goods:*

In [7]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db)

*Add to main database:*

In [8]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

#### Add module

The model class ```WasteManagementCGE``` contains the default specifications for the production module. We can adjust these using appropriate kwargs, but the following are default options:
* Class: ```DynamicNCES_emission_ExoIntRC``` with adjustment costs, multiple outputs per firm, waste generation, and *internal recycling* (without passing it to waste treatment firm).
* Use the lump sum tax in calibration of tax transfers in the baseline year.
* Initialize as a "partial" module: This allows us to first solve the model without general equilibrium conditions and then subsequently with (with a solution close to the "true").

The only things that we need to provide (for a default specification) are:
* Initialization relies on some dummies specifying waste generation - we add these to the database of the nesting tree.
* ```exoP```: The set of prices that are exogenous in the partial equilibrium model (and endogenous in general).
* ```exoQS```: The set of quantities of supply that are exogenous in partial equilibrium (and endogenous in general).

In [9]:
nestAgg.db[nestAgg.n('output')] = adj.rc_pd(db('qS'), nestAgg.db('s')).xs(t0).index.remove_unused_levels() # update set of outputs 
[nestAgg.db.__setitem__(k, db(k)) for k in ('dWTn','dWTy','dWTyF')]; # 
M.stdProduction(nestAgg,  exoP = db('d_pEqui'), exoQS = db('d_qSEqui'));

### B. Waste Management

The waste management sector are in some ways similar to the production firms, but, on top of the "usual" production structure, they supply waste treatment services that are modelled non-standard. 

#### Nesting structure

*Temporarily add a branch to the output structure: This is to make sure that the ZW node is designated an intermediate input.*

In [10]:
outTree = db('nestWasteOut').union(pd.MultiIndex.from_arrays([db('sWaste'), ['DELBRANCH'], ['ZW']], names = ['s','n','nn']))
nestInp = nestingTree.Tree('WInp', tree = db('nestWasteInp').to_list(), f = 'CES_scaled')
nestOut = nestingTree.Tree('WOut', tree = outTree.to_list(), f= 'CET_scaled')
nestAgg = nestingTree.AggTree(name = 'W', trees = {t.name: t for t in [nestInp, nestOut]}, ws = ws)
nestAgg(namespace = {str(n)+'_input': n for n in db('n').union(db('nm_D'))})

For the waste management firm, the production structure of the intermediate good ```KELM``` is straightforward. However, when this is split into the generic service ```Waste``` and ```ZW``` (treatment aggregate), we deviate from the standard production module methods. We now do the following:
* Remove ```DELBRANCH``` from nesting structure: This is only included to make sure that ```ZW``` is designated an "intermediate" input and assigned to the correct subsets/mappings.
* We set the price of ```ZW = 1``` and compute quantity from the waste treatment production function.
* We know the quantity supplied of the residual service ```Waste```. Given that the production structure is CRS, we then back out an initial guess of a cost-index of this ```Waste``` output.
* Given these inputs, we can use "standard" methods to get share parameters, prices, and quantities for the nesting structure that solves the model in the baseline. 
* Remove ```ZW``` from the set of "intermediate goods": This subset is used to guide what prices/quantities are endogenous/exogenous in what solution states. We need new definitions for this specific group. Furthermore, remove ```ZW``` from the set of knots that require a price index equation (outputs, knots):

*Remove DELBRANCH from nesting trees:*

In [11]:
delIdx = pd.Index(['DELBRANCH'], name = 'n')
[nestAgg.db.__setitem__(k, adj.rc_pd(nestAgg.db(k), ('not', delIdx))) for k in nestAgg.db.varDom('n', types = ['set','subset','map'])['n']]; 
[nestOut.db[k].__setattr__('vals', adj.rc_pd(nestOut.db[k], ('not', delIdx))) for k in nestOut.db if 'n' in nestOut.db[k].domains];
delIdx = pd.Index(['DELBRANCH'], name = 'nn')
[nestAgg.db.__setitem__(k, adj.rc_pd(nestAgg.db(k), ('not', delIdx))) for k in nestAgg.db.varDom('n', types = ['set','subset','map'])['nn']];
[nestOut.db[k].__setattr__('vals', adj.rc_pd(nestOut.db[k], ('not', delIdx))) for k in nestOut.db if 'nn' in nestOut.db[k].domains];

*Get initial values*

In [12]:
v = InitTree(nestAgg, ws = ws)

*Price target of 1 in $ZW$ nest*

In [13]:
zwIdx = adj.rc_pd(nestAgg.get('int'), db('n_ZW'))
pDTemp = pd.Series(1, index = zwIdx)

*Production of $ZW$ from waste treatment production function:*

In [14]:
qZW = (db('WTD_W') * (db('WTD_gd') * db('WTD_d') + db('WTD_ge') * db('WTD_e') + db('WTD_gr') * db('WTD_r')*(1-db('WTD_dmin')))).xs(t0).sum()
qDTemp = pd.Series(qZW, index = zwIdx)

*Production of residual service, back out balancing price index:*

In [15]:
qS, qD, pD = v.defaultQS(db), v.defaultQD(db), v.defaultPD(db)
pS = ((qD*pD).groupby('s').sum() - qZW) / qS

*Get initial values of $\mu$-parameters and intermediate goods:*

In [16]:
qD = qDTemp.combine_first(qD)
pD = pDTemp.combine_first(pD)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db, qD = qD, pD = pD, qS = qS, pS = pS, balancePS = False)

*Replace definition of outputs, remove ```ZW``` from a few sets:*

In [17]:
nestAgg.db[nestAgg.n('output')] = adj.rc_pd(db('qS'), db('sWaste')).xs(t0).index.remove_unused_levels()
[nestAgg.db[nestAgg.n(k)].__setattr__('vals', adj.rc_pd(nestAgg.get(k), ('not', db('n_ZW')))) for k in ('int','knout')];
[nestAgg.db[nestAgg.n(k, local = 'WOut')].__setattr__('vals', adj.rc_pd(nestAgg.get(k,local='WOut'), ('not', db('n_ZW')))) for k in ['knot']];

*Add to main database:*

In [18]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

#### Add module

In [19]:
[nestAgg.db.__setitem__(k, db(k)) for k in ('dWTn','dWTy','dWTyF','n_ZW')];
M.stdWasteManagement(nestAgg,  exoP = db('d_pEqui'), exoQS = db('d_qSEqui'));

### C. Investment

#### Nesting structure

In [20]:
nest = nestingTree.AggTree(name = 'I', trees = {'I': nestingTree.Tree('I', tree = db('nestInvest').to_list(), f = 'CES_scaled')}, ws = ws)
nest(namespace = {str(n)+'_input':n for n in db('n')});

*Get initial values of $\mu$-parameters and intermediate goods:*

In [21]:
v = InitTree(nest, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db)

*Add to main database:*

In [22]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

#### Add module

The production of investment goods follow a simple nested CES function as well. The model class here uses the following default options:
* Class: ```StaticNECS``` without emissions.
* Use the tax on outputs to calibrate total tax transfers in the baseline year.
* Initialize as a "partial" module: This allows us to first solve the model without general equilibrium conditions and then subsequently with (with a solution close to the "true").

Thus, if we only add the nesting tree, the other options are build in:

In [23]:
M.stdInvestment(nest);

### D. Consumers

In [24]:
nest = nestingTree.Tree('C', tree = db('nestHH').to_list(), f = 'CES_scaled')
nestAgg = nestingTree.AggTree(name = 'C', trees = {t.name: t for t in [nest]}, ws = ws)
nestAgg(namespace = {str(n)+'_input':n for n in db('n').union(db('nm_D'))})

*Get initial values of $\mu$-parameters and intermediate goods:*

In [25]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
# v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db)

*Add pS/qS for the top nest to the qD/pD vectors:*

In [26]:
[vals.__setitem__(f'{k}D', vals[f'{k}D'].combine_first(vals[f'{k}S'])) for k in ('q','p')];

*Add to main database:*

In [27]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

In [28]:
[nestAgg.db.__setitem__(k, db(k)) for k in ('dWTn','dWTy','dWTyF','t0')]; # symbols used in initialization
M.stdHousehold(nestAgg, db('L2C'));

### E. Government

Nesting tree:

In [29]:
cesNest = nestingTree.Tree('G', tree = db('nestG').to_list(), f = 'CES_scaled')
nest = nestingTree.AggTree(name = 'G', trees = {t.name: t for t in [cesNest]}, ws = ws)
nest(namespace = {str(n)+'_input': n for n in db('n')});

*Get initial values of $\mu$-parameters and intermediate goods:*

In [30]:
v = InitTree(nest, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
# v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db)

*Add pS/qS for the top nest to the qD/pD vectors:*

In [31]:
[vals.__setitem__(f'{k}D', vals[f'{k}D'].combine_first(vals[f'{k}S'])) for k in ('q','p')];

*Add to main database:*

In [32]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

The government sector is modelled with a nested CES function to determine demand from other sectors', and balances the budget using taxes on households. The model class here uses the following default options:
* Class: ```StaticNECS```.
* It uses the lump-sum tax to calibrate total tax transfers in the baseline year.
* It uses the lump-sum tax on households to ensure a balanced budget in all years.
* It uses intial level of assets to target total expenditures in the baseline year.
* Initialize as a "partial" module: This allows us to first solve the model without general equilibrium conditions and then subsequently with (with a solution close to the "true").

Simply specifying the nesting structure adopts the assumptions above as standard:

In [33]:
nest.db['t0'] = db['t0'] # this is used in the initialization phase - before merging the databases
M.stdGovernment(nest);

### F. Trade

The Armington trade module relies on two main specifications: Mapping from domestic to foreign types of goods (stored as ```dom2for``` in the database), and ```dExport``` that specifies the types of goods that the foreign sector demands. The ```Armingtong_waste``` extension simply adds that *waste imports* follow the same Armington specification as the *export* of waste treatment services (in our model the two are always tied directly together in this fashion).

All relevant symbols are already included in the main database, so we simply have to specify the name of the module:

In [34]:
M.stdTrade('T')
# M.stdTrade('T', cl = 'Armington', initFromGms = 'initArmingtonParams');

### G. Inventory investments

We include inventory investments for completeness, but simply keep them at an exogenous level for now. The default option uses the sector ``` s = 'itory'``` as the inventory investment sector:

In [35]:
M.stdInventory('IVT');

### G. Emissions

The default option here is to include emissions and taxes on CO2, but not include abatement technologies. For this, we only need to provide a name for the module: 

In [36]:
M.stdEmissions('M');

## 2. Prepare database

*Clean up database a bit (this is not necessary, but it removes some variables that are not ultimately used in the model):*

In [37]:
[db.series.__delitem__(k) for k in ('vD','vTax', 'vD_dur','vD_depr','vD_inv', 'vS', 'pD_dur') if k in db.symbols];

For variables that are defined over $t$, but where we do not yet have an initial value for all $t$, extrapolate from data:

*Note: This forces extrapolation of all variables defined over $t$ - if it is important that some variables are not extrapolated, they should be removed from this statement.*

In [38]:
[symbol.__setattr__('vals', extrapolateUpper(symbol.vals, db('tE')[0])) for symbol in [db[k] for k in db.varDom('t')['t']]];

Redefine sets based on what are actually used in variables, parameters, mapping, and subsets. Then merge internally, i.e. write a gdx file from the Python database:

In [39]:
AggDB.updSetsFromSyms(db, types = ['var','par','map','subset'], clean = False)  # clean = False means that we do not initially empty the symbols
db.mergeInternal() # write gdx file

## 3. Calibrate model

Calibrate the model without equilibrium constraints yet (the jSolve method automatically uses all the modules that we have added so far):

In [41]:
soldb = M.jSolve(5, state = 'C', ϕ = .5) # solve calibration model with 5 steps and nonlinear grid (ϕ<1 means that adjustments to jTerms start large and then decrease)

Write solution to the main database again:

In [42]:
[M.db.__setitem__(k, soldb[k]) for k in M.db.getTypes(['var']) if k in soldb.symbols]; # use solution database
M.db.mergeInternal()

Remove "init" methods and set state to general equilibrium (not partial):

In [43]:
[m.__setattr__('initFromGms', None) for m in M.m.values() if hasattr(m, 'initFromGms')]; # remove the GAMS initialization part
[m.__setattr__('partial', False) for m in M.m.values() if hasattr(m, 'partial')]; # remove partial eq. settings for now also.

Add equilibrium module and calibrate:

In [44]:
M.stdEquilibrium('Equi')
fullSol = M.jSolve(5, state = 'C')

Add full solution to baseline model database:

In [45]:
[M.db.__setitem__(k, fullSol[k]) for k in M.db.getTypes(['var']) if k in fullSol.symbols]; # use solution database
M.db.mergeInternal()

## 4. Save/export

In [46]:
M.db.data_folder = d['data']

We can save/export the model in a few different ways:
1. Save model instance: ```Model``` (ultimate parent class for all the models) is pickleable, meaning that we can save/load the class with Python's ```pickle``` class.
2. Store solution: It takes no virtually no time to re-compile the ```Model``` class, so we can also simply store the solution database and - when needed - initialize the model class again.
3. Store GAMS code and solution: This option allows us to remove the model from the python class ```Model``` and instead treat it as a conventional GAMS program.

*1. Store model instance or database with pickle:*

In [51]:
M.export(repo = d['data'], name = M.name) # store the entire class - these are the default options by the way 

*2. Store data:*

In [50]:
M.db.export(repo = d['data'], name = M.db.name) # store the solution database  - these are the default options by the way 